# 00 · Tests unitarios

| | |
|---|---|
| **Objetivo** | Verificar la logica pura del proyecto antes de gastar computo en el pipeline |
| **Entradas** | `src/nyc_taxi/` y `tests/unit/` |
| **Salidas** | Reporte de ejecucion. No produce ni modifica datos |
| **Depende de** | Nada. Es el primer notebook a ejecutar |

**Proceso**
1. Instalar pytest en el entorno de la sesion
2. Localizar la raiz del repositorio de forma dinamica
3. Ejecutar la suite completa
4. Fallar el notebook si algun test no pasa

Las pruebas cubren geometria, reglas de validacion, tipado, split, metricas
y deteccion de deriva. Son logica pura, sin sesion de Spark, y corren en
menos de un segundo.

In [0]:
%pip install pytest --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import os
import subprocess
import sys

# El sistema de archivos de /Workspace no permite crear directorios __pycache__,
# y pytest intenta escribir ahí el caché de reescritura de asserts. Desactivar la
# escritura de bytecode evita el OSError [Errno 95] Operation not supported.
ENTORNO = {**os.environ, "PYTHONDONTWRITEBYTECODE": "1"}


# Misma localización dinámica que usan los demás notebooks: sube desde el
# directorio actual hasta encontrar la carpeta que contiene src/. Nada de rutas
# fijas, para que el repo pueda vivir a cualquier profundidad del workspace.
def raiz_repo(marcador="src", max_niveles=10):
    ruta = os.getcwd()
    for _ in range(max_niveles):
        if os.path.isdir(os.path.join(ruta, marcador)):
            return ruta
        padre = os.path.dirname(ruta)
        if padre == ruta:
            break
        ruta = padre
    raise RuntimeError(
        f"No se encontró la raíz del repo (carpeta con '{marcador}/') "
        f"partiendo de {os.getcwd()}."
    )


RAIZ = raiz_repo()
print(f"Raíz del repo: {RAIZ}")

Raíz del repo: /Workspace/Users/fernandogomez0621@gmail.com/nyc-taxi-trip-duration (20)


In [0]:
resultado = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "--tb=short", "-p", "no:cacheprovider"],
    cwd=RAIZ,
    env=ENTORNO,
    capture_output=True,
    text=True,
)

print(resultado.stdout)
if resultado.stderr:
    print("--- stderr ---")
    print(resultado.stderr)

============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-8.3.5, pluggy-1.5.0 -- /local_disk0/.ephemeral_nfs/envs/pythonEnv-cf89b138-0dd8-44cb-a6e5-3fe4dde8c7fb/bin/python
rootdir: /Workspace/Users/fernandogomez0621@gmail.com/nyc-taxi-trip-duration (20)
configfile: pyproject.toml
testpaths: tests
plugins: langsmith-0.6.1, anyio-4.7.0
collecting ... collected 85 items

tests/unit/test_data_prep.py::TestValidaciones::test_duracion[0-False] PASSED [  1%]
tests/unit/test_data_prep.py::TestValidaciones::test_duracion[1-True] PASSED [  2%]
tests/unit/test_data_prep.py::TestValidaciones::test_duracion[455-True] PASSED [  3%]
tests/unit/test_data_prep.py::TestValidaciones::test_duracion[21600-True] PASSED [  4%]
tests/unit/test_data_prep.py::TestValidaciones::test_duracion[21601-False] PASSED [  5%]
tests/unit/test_data_prep.py::TestValidaciones::test_duracion[86400-False] PASSED [  7%]
tests/unit/test_data_prep.py::TestValidacione

## Resultado

El `assert` hace que el notebook falle si algún test falla, de modo que
pueda encadenarse como primera tarea de un job y detener el pipeline antes
de tocar el dato.

In [0]:
assert resultado.returncode == 0, (
    f"La suite de tests falló (código {resultado.returncode}). "
    "Revisar la salida de la celda anterior."
)

print("Todos los tests pasaron.")

Todos los tests pasaron.
